# Practical 5 - Part 2. 

In this part of the practical we will use the trained keypoint detector to localise the microplate and then grasp it using our motion planner.

Before you start, check whether a Robotiq 2F85 gripper is attached to the robot. Power on your robot, make sure your robot is first in **local control mode** and complete the following task:
<div class="alert alert-block alert-info"> <b>Task:</b> Configure the robot TCP length and payload. The tool length is 175 mm when closed. The tool's weight is 0.9 kg. When finished, set your robot to remote control mode.</div>

Configured? Now go to **remote control mode**

## Setup

<div class="alert alert-block alert-info"> <b>Task</b> Before we get started, run the next cells to install all necessary packages and dependencies. Afterwards, you can comment them so you do not re-run them when you have to reload the kernel.
 </div>

In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

In [ ]:
!pip uninstall -y airo-models
!pip install airo-models
!pip install airo-planner 

In [ ]:
# SKIP THIS CELL FOR NOW 
# Install OMPL 
# if you have Python 3.10 uncomment the following:
#!pip install https://github.com/ompl/ompl/releases/download/prerelease/ompl-1.6.0-cp310-cp310-manylinux_2_28_x86_64.whl
# if you have Python 3.11 uncomment the following
#!pip install https://github.com/ompl/ompl/releases/download/prerelease/ompl-1.6.0-cp311-cp311-manylinux_2_28_x86_64.whl

In [ ]:
# Install our analytical IK library for the UR robots. You can comment this line once you installed it.  
!pip install ur_analytic_ik 

In [ ]:
# We will first use conda to install pytorch (CPU version - inference only) to avoid having the keypoint detector library install it over pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
#install the keypoint detector 
!git clone https://github.com/tlpss/keypoint-detection.git
! cd keypoint-detection && pip install .

In [1]:
# Some group their installations historically had problems with some libraries. We are keeping these instructions here in case your group run into similar problems.
# Some downgrade hacks to fix pytorch lightning
#!pip install matplotlib==3.3
# At time of running this practical in dec 2024, the latest version of wandb has some login problems
#!pip install wandb==0.18.7

<div class="alert alert-block alert-info"> <b>Task</b>  Run the following cell to log in to wandb.
 </div>

In [ ]:
# log in to wandb.
!wandb login

# If you cannot paste your key in the jupyter notebook output cell, use and uncomment the following cell

In [ ]:
#import wandb
#wandb.login()

## Using the Keypoint Model to Determine Microplate Frame

In this section we will use the trained keypoint detector to localise the microplate, so that we no longer have to manually determine a number of points. 

This requires the following steps:

- load the trained model
- calibrating the camera extrinsics
- define two views for the robot to observe the microplate
- use the trained model to detect the keypoints on those images
- triangulate these points
- localise the frame of the microplate based on the 3D points

we will tackle them one by one.

In [ ]:
## preparing the model for local inference

import torch
import keypoint_detection
from keypoint_detection.utils.load_checkpoints import get_model_from_wandb_checkpoint
from keypoint_detection.utils.heatmap import get_keypoints_from_heatmap_batch_maxpool
from keypoint_detection.utils.visualization import draw_keypoints_on_image
from keypoint_detection.models.detector import KeypointDetector
import numpy as np
from PIL import Image

<div class="alert alert-block alert-info"> <b>Task</b>  fill in the path to the model you trained in part one. This should look like "tlips/irm-microplate-keypoints/model-4o9lkyqg:v0".
 </div>

In [ ]:
#TODO(@Student): Fill in model full name from wandb here. You should be logged in into wandb or set your project to public.
model = get_model_from_wandb_checkpoint("")

<div class="alert alert-block alert-info"> <b>Task</b> Now run the next cells to test the model on an example image. It should detect exactly one point for each of the four keypoints.
 </div>

In [ ]:
def run_inference(model: KeypointDetector, image: Image, confidence_threshold: float = 0.05):
    """ image: (H,W,C) ints in range 0-255"""
    model.eval()
    tensored_image = torch.from_numpy(np.array(image)).float()
    tensored_image = tensored_image / 255.0
    tensored_image = tensored_image.permute(2, 0, 1)
    tensored_image = tensored_image.unsqueeze(0)
    with torch.no_grad():
        heatmaps = model(tensored_image)

    keypoints = get_keypoints_from_heatmap_batch_maxpool(heatmaps, abs_max_threshold=confidence_threshold,
                                                         max_keypoints=1)
    image_keypoints = keypoints[0]

    for keypoints, channel_config in zip(image_keypoints, model.keypoint_channel_configuration):
        print(f"Keypoints for {channel_config}: {keypoints}")
    return image_keypoints


img_dummy = Image.open('example-image.png')
np.asarray(img_dummy).shape
dummy_image_keypoints = run_inference(model, img_dummy)

In [ ]:
import matplotlib.pyplot as plt


def visualize_predicted_keypoints(model, keypoints, image):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)
        
    img_with_kps = draw_keypoints_on_image(image, keypoints, model.keypoint_channel_configuration)
    # plt.imshow(img_with_kps)
    return img_with_kps


visualize_predicted_keypoints(model, dummy_image_keypoints, img_dummy)

<div class="alert alert-block alert-info"> <b>Task</b> Now put the microplate on the bottom shelve similar as on the image above. Your robot will have to be able to grasp the microplate so choose an appropriate location. You can verify this position by freedriving the arm to the microplate.
 </div>


<div class="alert alert-block alert-info"> <b>Task</b> Run the camera calibration as in the last practical.. Save the pose as `camera_pose.json`. Then run the next cell to load the pose.
 </div>

When in doubt, take a look at the [instructions](https://github.com/airo-ugent/airo-mono/blob/main/airo-camera-toolkit/airo_camera_toolkit/calibration/README.md).


In [ ]:
from airo_camera_toolkit.cameras.realsense.realsense import Realsense

camera = Realsense()

In [ ]:
!airo-camera-toolkit hand-eye-calibration --robot_ip "10.42.0.162" --mode eye_in_hand

In [ ]:
from airo_dataset_tools.data_parsers.pose import Pose

camera_pose_path = "camera_pose.json"

with open(camera_pose_path, "r") as f:
    camera_pose_in_tcp = Pose.model_validate_json(f.read()).as_homogeneous_matrix()

camera_pose_in_tcp

<div class="alert alert-block alert-info"> <b>Task</b> Run the cell below to instantiate the robot, gripper and camera interfaces.
 </div>

In [ ]:
from airo_camera_toolkit.cameras.realsense.realsense import Realsense
from airo_robots.manipulators.hardware.ur_rtde import URrtde
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85

ip_address = "10.42.0.162"  # fill in the robot IP
robot = URrtde(ip_address, URrtde.UR3E_CONFIG)
gripper = Robotiq2F85(ip_address)

<div class="alert alert-block alert-info"> <b>Task</b> Now define a good home position for the robot. Preferably the robot TCP is located between its base and the shelve and points down to the table. You can put the robot in freedrive and find a suitable configuration. Use the controller to read the joint positions and fill them in below. 
Run the two cells below to move the robot to the home position.
  </div>

In [ ]:
robot.rtde_control.teachMode()

In [ ]:
robot.get_joint_configuration()

In [ ]:
robot.rtde_control.endTeachMode()

In [ ]:
#TODO(@Students): fill in home position (radians!)
joints_home = np.array([])

In [ ]:
robot.move_to_joint_configuration(joints_home).wait()

<div class="alert alert-block alert-info"> <b>Task:</b> Now define two joint configurations that enable the camera to observe the entire bottom shelve of the rack using the same procedure as before. Then run the three cells below to move the robot to these configurations and capture two images of the bottom shelve and the microplate.
 </div>

<div class="alert alert-block alert-warning"> Make sure the minimum distance between the camera eye and the microplate is larger than 30 cm! </div>

You can use the following three cells to freedrive the robot and get its configuration and camera image to copy paste in the fourth cell from here. 


In [ ]:
robot.rtde_control.teachMode()

In [ ]:
print(np.array2string(robot.get_joint_configuration(),separator=', ', precision=4))
plt.imshow(camera.get_rgb_image_as_int())

In [ ]:
robot.rtde_control.endTeachMode()

In [ ]:
#TODO(@Students): Fill in your joint configurations for taking photos of the uplate
joints_view_A = np.array()
joints_view_B = np.array()

In [ ]:
# move there with the robot and collect images
import time


def collect_rgbd_and_tcp_pose(robot, camera, joints):
    robot.move_to_joint_configuration(joints, joint_speed=1 ).wait()
    time.sleep(1.0)
    image = camera.get_rgb_image_as_int()
    depth_map = camera._retrieve_depth_map()
    X_B_TCP = robot.get_tcp_pose()
    return image, depth_map, X_B_TCP


def imshow_left_right(image_left, image_right):
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(image_left)
    plt.title("Left image")
    plt.subplot(1, 2, 2)
    plt.imshow(image_right)
    plt.title("Right image")
    plt.show()


image_A, depth_map_A, X_B_TCPA = collect_rgbd_and_tcp_pose(robot, camera, joints_view_A)
image_B, depth_map_B, X_B_TCPB = collect_rgbd_and_tcp_pose(robot, camera, joints_view_B)

In [ ]:
imshow_left_right(image_A, image_B)

We have trained the keypoint detector on images of  1024x512 pixels. We hence need to resize the two camera images to the same resolution, before passing them through the network. 
<div class="alert alert-block alert-info"> <b>Task:</b> Run the cell below to create and apply this transform.  </div>


In [ ]:
# resize the images
from airo_camera_toolkit.image_transforms.transforms.resize import Resize

transform = Resize(image_A.shape, h=512, w=1024)

image_A_transformed = transform.transform_image(image_A)
image_B_transformed = transform.transform_image(image_B)


<div class="alert alert-block alert-info"> <b>Task:</b>  Run the cells below to use the model to detect the keypoints on the images. Make sure that the detector predicts a point for every semantic location in both images! Evaluate the result of your keypoint detection well. Are there for both images 4 keypoints detected? Are they +- accurate? What can you do if your keypoint detector did not find all 4 keypoints? Write your strategies down here and rerun the cells above until you get good keypoints. 
</div>
<div class="alert alert-block alert-warning"> NOTE you should not spend more than 20 minutes on this. If you cannot get your keypoint detector working after 20 minutes, you can continue the practical by manually labeling the keypoints as you did in practical 4. Just make sure to write down your process (i.e. the things you have tried) in the next cell and continue with the practical.  </div>

In [ ]:
keypoints_A = run_inference(model, image_A_transformed)
keypoints_B = run_inference(model, image_B_transformed)

In [ ]:
visualize_predicted_keypoints(model, keypoints_A, image_A_transformed)

In [ ]:
visualize_predicted_keypoints(model, keypoints_B, image_B_transformed)

TODO(@Students): 
write down your process of helping the keypoint detector to detect 4 keypoints.  It can be useful to use the helper cell below to detect keypoints live while moving the arm. If you run the next cell, you have to cancel it by interrupting the kernel (i.e., the stop button in your jupyter notebook UI).


In [ ]:
# This cell helps you to find keypoints with your detector live. It helps you debugging. Abort this cell by interrupting the kernel using jupyter notebook UI.
import cv2 
from IPython.display import clear_output, display
import numpy as np
import time 

# Begin teach mode
robot.rtde_control.teachMode()

try:
    while True:
        # Capture an image
        image_A = camera.get_rgb_image_as_int()
        print(np.array2string(robot.get_joint_configuration(),separator=', ', precision=4))

        # Transform and run inference on the image
        transformed_image = transform.transform_image(image_A)
        inference_result = run_inference(model, transformed_image)
        visualization = visualize_predicted_keypoints(model, inference_result, transformed_image)
        
        # Display the result
        clear_output(wait=True)
        display(visualization)
        
        time.sleep(0.1)
finally:
    # End teach mode
    robot.rtde_control.endTeachMode()

<div class="alert alert-block alert-info"> <b>Task:</b> Set the following boolean variable whether you will use the keypoint detector in the remainder of this practical. </div> We use this so you avoid running cells that overwrite the result of your keypoint detector.

In [ ]:
USE_KP_DETECTOR = True

We also have to transform the keypoints back to the original image size, so that we can use them for triangulation.
<div class="alert alert-block alert-info"> <b>Task:</b> Run the cell below to reverse transform the keypoint-detected image to its native image size. </div>

In [ ]:
if USE_KP_DETECTOR:
    keypoints_A_original = [transform.reverse_transform_point(x[0]) for x in keypoints_A]
    keypoints_B_original = [transform.reverse_transform_point(x[0]) for x in keypoints_B]

In case your keypoint detector did not work, use the following code the annotate the keypoints manually.

In [ ]:
from typing import Tuple

import cv2
from airo_camera_toolkit.utils.image_converter import ImageConverter
from airo_typing import (
    CameraIntrinsicsMatrixType,
    HomogeneousMatrixType,
    NumpyDepthMapType,
    NumpyIntImageType,
    Vector2DType,
    Vector3DType,
    JointConfigurationType,
)

if not USE_KP_DETECTOR:
    
    cyan = (255, 255, 0)  # default color for left image
    magenta = (255, 0, 255)  # default color for right image
    
    cv_white = (255,255,255)
    cv_black = (20,20,20)
    cv_yellow = (30, 255, 255)
    cv_purple = (255, 0, 255)
    
    def draw_point_coordinate(image, point, color):
        """Draws a circle on the image at the given point with the coordinates next to it."""
        text = f"({point[0]}, {point[1]})"
        text_location = (point[0] + 12, point[1] - 12)
        cv2.circle(image, point, 1, color, 1)
        cv2.circle(image, point, 15, color, 4)
        cv2.putText(image, text, text_location, cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3, cv2.LINE_AA)
    
    
    def annotate_multiview(
        left_image: NumpyIntImageType,
        right_image: NumpyIntImageType,
        color_left: Tuple[int, int, int] = cyan,
        color_right: Tuple[int, int, int] = magenta,
    ):
        left_point = [None]
        right_point = [None]
    
        def on_click_left(event, x, y, flags, param):
            if event == cv2.EVENT_LBUTTONDOWN:
                print(f"left click: (u,v) =  ({x}, {y})")
                left_point[0] = (x, y)
    
        def on_click_right(event, x, y, flags, param):
            if event == cv2.EVENT_LBUTTONDOWN:
                print(f"right click: (u,v) =  ({x}, {y})")
                right_point[0] = (x, y)
    
        cv2.namedWindow("Left Image", cv2.WINDOW_NORMAL)
        cv2.namedWindow("Right Image", cv2.WINDOW_NORMAL)
        cv2.setMouseCallback("Left Image", on_click_left)
        cv2.setMouseCallback("Right Image", on_click_right)
    
        # resize the windows and place them side by side
        cv2.resizeWindow("Left Image", 800, 600)
        cv2.resizeWindow("Right Image", 800, 600)
        cv2.moveWindow("Left Image", 0, 0)
        cv2.moveWindow("Right Image", 800, 0)
    
        left_image_bgr = ImageConverter.from_numpy_int_format(left_image).image_in_opencv_format
        right_image_bgr = ImageConverter.from_numpy_int_format(right_image).image_in_opencv_format
    
        print("Press 'Enter' or 'q' to finish.")
        while True:
            left_annotated = left_image_bgr.copy()
            right_annotated = right_image_bgr.copy()
    
            if left_point[0] is not None:
                draw_point_coordinate(left_annotated, left_point[0], color_left)
    
            if right_point[0] is not None:
                draw_point_coordinate(right_annotated, right_point[0], color_right)
    
            cv2.imshow("Left Image", left_annotated)
            cv2.imshow("Right Image", right_annotated)
            key = cv2.waitKey(1)
    
            if key == ord("q") or key == 13:
                break
    
        print("Left point: ", left_point[0])
        print("Right point: ", right_point[0])
        cv2.destroyAllWindows()
        return left_point[0], right_point[0]
    
    
    keypoint_bottom_right_A, keypoint_bottom_right_B = annotate_multiview(image_A, image_B, color_left=cv_white, color_right=cv_white)
    keypoint_bottom_left_A, keypoint_bottom_left_B = annotate_multiview(image_A, image_B, color_left=cv_black, color_right=cv_black)
    keypoint_top_left_A, keypoint_top_left_B = annotate_multiview(image_A, image_B, color_left=cv_yellow, color_right=cv_yellow)
    keypoint_top_right_A, keypoint_top_right_B = annotate_multiview(image_A, image_B, color_left=cv_purple, color_right=cv_purple)

    keypoints_A_original = [ [keypoint_bottom_right_A] , [keypoint_bottom_left_A] , [keypoint_top_left_A], [keypoint_top_right_A]]
    keypoints_B_original = [ [keypoint_bottom_right_B] , [keypoint_bottom_left_B] , [keypoint_top_left_B], [keypoint_top_right_B]]

In [ ]:
from airo_typing import HomogeneousMatrixType, CameraIntrinsicsMatrixType, Vector2DType


def least_squares_triangulation(
        point_left: Vector2DType,
        point_right: Vector2DType,
        intrinsics: CameraIntrinsicsMatrixType,
        left_extrinsics: HomogeneousMatrixType,
        right_extrinsics: HomogeneousMatrixType,
) -> np.ndarray:
    """Triangulates a 3D point from a pair of 2D points in the left and right image

    Args:
        point_left: (u,v) pixel coordinate of the point in the left image
        point_right: (u,v) pixel coordinates of the point in the right image
        intrinsics: the camera's intrinsics matrix
        left_extrinsics: camera pose for the left image
        right_extrinsics: camera pose for the right image (must be expressed in the same frame as the left pose])

    Returns:
        the 3D point (expressed in the same frame as the camera poses)
    """

    def _to_homogeneous(np_array):
        return np.append(np_array, 1)

    # create the rays in the camera frome from the pixel coordinates
    ray_left = np.linalg.inv(intrinsics) @ _to_homogeneous(np.array(point_left))
    ray_right = np.linalg.inv(intrinsics) @ _to_homogeneous(np.array(point_right))

    # transform the rays to the world frame
    ray_left_in_world = left_extrinsics[:3, :3] @ ray_left
    ray_right_in_world = right_extrinsics[:3, :3] @ ray_right

    # reshape the rays to be column vectors
    ray_right_in_world = ray_right_in_world.reshape(3, 1)
    ray_left_in_world = ray_left_in_world.reshape(3, 1)

    # create the least squares LHS and RHS
    lhs = 0
    rhs = 0
    for ray, camera_position in zip(
            (ray_left_in_world, ray_right_in_world), (left_extrinsics[:3, 3], right_extrinsics[:3, 3])
    ):
        lhs += np.eye(3) - ray @ ray.T / np.linalg.norm(ray) ** 2
        rhs += (np.eye(3) - ray @ ray.T / np.linalg.norm(ray) ** 2) @ camera_position

    return np.linalg.inv(lhs) @ rhs

<div class="alert alert-block alert-info"> <b>Task: </b> In the cell below, triangulate the 4 keypoints. Make use of the `least_squares_triangulation` method. </div>

In [ ]:
X_B_CA = X_B_TCPA @ camera_pose_in_tcp
X_B_CB = X_B_TCPB @ camera_pose_in_tcp

points_3d = []
for keypoint_A, keypoint_B in zip(keypoints_A_original, keypoints_B_original):
    point_3D = ... #TODO @student: triangulate each point 

    points_3d.append(point_3D)

bottom_right = np.array(points_3d[0])
bottom_left = np.array(points_3d[1])
top_left = np.array(points_3d[2])
top_right = np.array(points_3d[3])

For the microplate we will use a frame as defined in the image below:

![](resources/microplate-frame.png)

<div class="alert alert-block alert-info"> <b>Task:</b> Fill in the cell below to define the pose of the microplate frame using the four 3D points. Try to make this as robust as possible by averaging over multiple points or directions.</div>


In [ ]:
#TODO @Student: make a 4x4 homogeneous matrix (as numpy array) that defines the pose of the microplate frame using the four 3D points
# You can use our SE3Container.from_orthogonal_base_vectors_and_translation(x_axis, y_axis, z_axis, origin).homogeneous_matrix function but this is not necessary. 
from airo_spatial_algebra.se3 import SE3Container

X_base_microplate = ... # homogeneous matrix of the microplate in the base frame

<div class="alert alert-block alert-info"> <b>Task:</b>  Run the cell below to visualize the frame in the pointcloud.
</div>

<div class="alert alert-block alert-warning"> You might get muddy results and notice the keypoints being projected with a wrong depth. This is due to your pointcloud. You can first try to use the pointcloud of your first camera view pose by changing the line commented with `#@Student: change variables with suffix B to suffix A` in the next cell. If those results are also bad, you better choose different viewing angles for your photos and make sure there is at least 30 cm between camera eye and microplate. 
</div>

In [ ]:
from typing import Optional, Tuple
import open3d as o3d
import numpy as np
from airo_typing import HomogeneousMatrixType, CameraIntrinsicsMatrixType, NumpyDepthMapType, NumpyIntImageType, \
    Vector2DType, Vector3DType

from typing import Optional


def open3d_point(
    position: Vector3DType, color: Tuple[float, float, float], radius: float = 0.005
) -> o3d.geometry.TriangleMesh:
    """Creates a small sphere mesh for visualization in open3d.

    Args:
        position:  position of the point
        color: RGB color of the point
        radius: radius of the sphere

    Returns:
        sphere: a mesh
    """
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere.translate(position)
    sphere.paint_uniform_color(color)
    sphere.compute_vertex_normals()
    return sphere


def open3d_camera(
    camera_pose: HomogeneousMatrixType,
    intrinsics: CameraIntrinsicsMatrixType,
    resolution: Tuple[int, int],
    color: Optional[Tuple[float, float, float]] is None,
) -> o3d.geometry.LineSet:
    """Creates a camera visualization in open3d.

    Args:
        camera_pose: camera pose
        intrinsics: camera intrinsics
        resolution: width, height of the camera's images

    Returns:
        camera_lines: a line set
    """

    camera_lines = o3d.geometry.LineSet.create_camera_visualization(
        *resolution, intrinsics, np.linalg.inv(camera_pose), scale=0.1
    )
    if color is not None:
        camera_lines.paint_uniform_color(color)

    return camera_lines


def open3d_ray(direction, origin=np.array([0, 0, 0])):
    c = 0.001
    ray_mesh = o3d.geometry.TriangleMesh.create_arrow(c, 4 * c, 1.5, 5 * c)

    rotation_Z = direction
    rotation_X = np.cross(rotation_Z, np.array([0, 1, 0]))
    rotation_Y = np.cross(rotation_Z, rotation_X)
    rotation_matrix = np.column_stack([rotation_X, rotation_Y, rotation_Z])

    ray_mesh.rotate(rotation_matrix, center=(0, 0, 0))

    ray_mesh.translate(origin)

    return ray_mesh


def open3d_ray_from_pixel(
    pixel: Vector2DType,
    camera_pose: HomogeneousMatrixType,
    intrinsics: CameraIntrinsicsMatrixType,
    color: Optional[Tuple[float, float, float]] = None,
) -> o3d.geometry.TriangleMesh:
    """Creates a ray visualization in open3d.

    Args:
        pixel: pixel coordinate of the ray
        camera_pose: camera pose
        intrinsics: camera intrinsics

    Returns:
        ray_mesh: a triangle mesh
    """

    point_homogeneous = np.array([pixel[0], pixel[1], 1])
    ray_direction = np.linalg.inv(intrinsics) @ point_homogeneous
    ray_direction_in_world = camera_pose[:3, :3] @ ray_direction
    ray_origin_in_world = camera_pose[:3, 3]

    ray_mesh = open3d_ray(ray_direction_in_world, ray_origin_in_world)

    if color is not None:
        ray_mesh.paint_uniform_color(color)

    return ray_mesh


def open3d_pointcloud_from_image_and_depth_map(
    image: NumpyIntImageType,
    depth_map: NumpyDepthMapType,
    intrinsics: CameraIntrinsicsMatrixType,
    camera_pose: HomogeneousMatrixType,
    resolution: Tuple[int, int],
) -> o3d.geometry.PointCloud:
    image_o3d = o3d.geometry.Image(image.copy())
    depth_map_o3d = o3d.geometry.Image(depth_map)
    rgbd_o3d = o3d.geometry.RGBDImage.create_from_color_and_depth(
        image_o3d, depth_map_o3d, depth_scale=1.0, depth_trunc=100.0, convert_rgb_to_intensity=False
    )
    intrinsics_o3d = o3d.camera.PinholeCameraIntrinsic(resolution[0], resolution[1], intrinsics)

    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_o3d, intrinsics_o3d, extrinsic=np.linalg.inv(camera_pose)
    )
    return pcd


## open3D helper functions
def visualize_open3D(geometry_list):
    o3d.visualization.draw_geometries(
        geometry_list,
        zoom=0.2,
        front=[1.0, 0.0, 1.0],
        lookat=[0.0, -0.3, 0.0],
        up=[0.0, 0, 1.0],
    )



# visualize_open3D([pcd,  camera_A_lines, camera_B_lines])
# ===========================

yellow = (1.0, 1.0, 0.0)
cyan = (0.0, 1.0, 1.0)
magenta = (1.0, 0.0, 1.0)

camera_A_lines = open3d_camera(X_B_CA, camera.intrinsics_matrix(), camera.resolution, cyan)
camera_B_lines = open3d_camera(X_B_CB, camera.intrinsics_matrix(), camera.resolution, magenta)

#@Student: change variables with suffix B to suffix A (if your cloud point is of low quality)
# i.e.     image_B, depth_map_B, camera.intrinsics_matrix(), X_B_CB, camera.resolution
# to     image_A, depth_map_A, camera.intrinsics_matrix(), X_B_CA, camera.resolution
pcd = open3d_pointcloud_from_image_and_depth_map(
    image_B, depth_map_B, camera.intrinsics_matrix(), X_B_CB, camera.resolution
)

point_3D_mesh = [open3d_point(point, yellow) for point in points_3d]

microplate_frame_3d = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
microplate_frame_3d.transform(X_base_microplate)
visualize_open3D([pcd, camera_A_lines, camera_B_lines, microplate_frame_3d] + point_3D_mesh)

point_3D


## Grasping the microplate in a scene

To grasp the plate, we have to define a grasp pose in its frame. We also need to define a pregrasp pose (why?) and a retreat pose. 

For the grasp pose, we want the robot to grasp the object from the side. We will assume that the microplate is oriented with the orange sticker to the back of the the rack and want to approach it from the +X direction of its frame. 

<div class="alert alert-block alert-info"> <b>Task: </b> Why is it useful to define a pregrasp pose?</div>

TODO @Student: Your answer: 

<div class="alert alert-block alert-info"> <b>Task: </b> Define the grasp frame in the microplate frame and then transform it to determine the grasp frame in the base frame of the robot.</div>

In [ ]:
# assumes the plate is reachable by the left side for the robot! (on the shelve.)
#TODO @student: define the grasp frame
X_Microplate_TcpGrasp = ...
X_base_TcpGrasp = ... # remember that you already found X_base_microplate 

<div class="alert alert-block alert-info"> <b>Task:</b> Run the cell below to visualize the pose of the frame. Make sure it looks as expected. </div>

In [ ]:
grasp_frame_in_base_3d = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
grasp_frame_in_base_3d.transform(X_base_TcpGrasp)
visualize_open3D([pcd, camera_A_lines, camera_B_lines, grasp_frame_in_base_3d] + point_3D_mesh)

<div class="alert alert-block alert-info"> <b>Task:</b> In the cell below define a pregrasp frame (relative to the grasp frame) that is translated 10cm in the direction pointing outward from the rack (such that the gripper is standing in front of the microplate). The result will be visualized. Make sure the pregrasp frame is properly defined. </div>

In [ ]:
#TODO @student: define the pregrasp pose (i.e., stand "in front" of the microplate)
X_microplate_pregrasp = ...
X_base_pregrasp = ...


pregrasp_in_base_3d = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
pregrasp_in_base_3d.transform(X_base_pregrasp)
visualize_open3D([pcd,camera_A_lines,camera_B_lines,pregrasp_in_base_3d] + point_3D_mesh )

<div class="alert alert-block alert-info"> <b>Task:</b> Now do the same for the retreat frame (relative to the grasp frame): translate it 10cm in the direction pointing outward from the rack (again, relative to the grasp frame) (such that the gripper is standing in front of the microplate) and make it go 5 cm upwards (the Z direction of the world) to lift up the microplate. The result will be visualized. Make sure the pregrasp frame is properly defined. </div>

In [ ]:
#TODO @student: define the retreat pose (i.e., a lift-up movement after the grasp pose)

X_microplate_retreat = ...
X_base_retreat = ...


retreat_in_base_3d = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
retreat_in_base_3d.transform(X_base_retreat)
visualize_open3D([pcd,camera_A_lines,camera_B_lines,retreat_in_base_3d] + point_3D_mesh )

### Motion Planning

Now that we have all the required frames, we will use the planner to move the robot. 

<div class="alert alert-block alert-info"> <b>Task:</b> Why do we use a motion planner for this instead of robot.move_to_tcp_pose()? Answer below in your own words. </div>

Your answer: 

In [ ]:
import os
import random
import sys
from pathlib import Path
import numpy as np
import math
from loguru import logger

import airo_models
from airo_drake import finish_build
from airo_drake import visualize_frame
from airo_models.urdf import replace_value, make_static
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85
from airo_robots.manipulators.hardware.ur_rtde import URrtde
from airo_robots.manipulators import PositionManipulator
from pydrake.trajectories import PathParameterizedTrajectory
from pydrake.geometry import MeshcatVisualizer, MeshcatVisualizerParams, Role
from pydrake.math import RollPitchYaw
from pydrake.multibody.tree import FixedOffsetFrame
from pydrake.planning import RobotDiagramBuilder
from pydrake.geometry import Meshcat
from pydrake.math import RigidTransform, RotationMatrix
from kinematics import RobotKinematics
from airo_planner import SingleArmOmplPlanner
from pydrake.planning import SceneGraphCollisionChecker
from airo_drake import animate_joint_trajectory, time_parametrize_toppra, visualize_frame
from airo_typing import HomogeneousMatrixType, JointConfigurationType
from airo_dataset_tools.data_parsers.pose import Pose
from typing import List
from ur_analytic_ik_ext import ur3e

PLANNER_USE_ANALYTICAL_IK = False

# Change the minimum logging level
logger.remove()
logger.add(sys.stdout, format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
           filter=lambda record: record["level"].name == "INFO")


# Some helper functions. You can ignore these. 
def add_meshcat(robot_diagram_builder: RobotDiagramBuilder, meshcat=None) -> Meshcat:
    scene_graph = robot_diagram_builder.scene_graph()
    builder = robot_diagram_builder.builder()

    # Adding Meshcat must be done before finalizing
    meshcat = Meshcat() if meshcat is None else meshcat
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

    # Add visualizer for proximity/collision geometry
    collision_params = MeshcatVisualizerParams(role=Role.kProximity, prefix="collision", visible_by_default=False)
    MeshcatVisualizer.AddToBuilder(builder, scene_graph.get_query_output_port(), meshcat, collision_params)

    return meshcat


def _make_static(urdf):
    joint_types = ["revolute", "continuous", "prismatic", "floating", "planar"]
    for joint_type in joint_types:
        replace_value(urdf, "@type", joint_type, "fixed")
    return urdf

def inverse_kinematics_fn(X_B_TcpTarget: HomogeneousMatrixType) -> list[JointConfigurationType]:
    if PLANNER_USE_ANALYTICAL_IK:
        q_goal = _inverse_kinematics_analytical(X_B_TcpTarget)
    else:
        q_goal = _inverse_kinematics_optimization_relaxed(X_B_TcpTarget)
    return q_goal

def _inverse_kinematics_analytical(tcp_pose: HomogeneousMatrixType) -> List[JointConfigurationType]:
    X_Tool0Tcp = np.identity(4)
    X_Tool0Tcp[2, 3] = 0.175

    solutions = ur3e.inverse_kinematics_with_tcp(tcp_pose, X_Tool0Tcp)
    solutions = [solution.squeeze() for solution in solutions]
    return solutions

def _inverse_kinematics_optimization_relaxed(X_B_TCP_target: HomogeneousMatrixType) -> List[JointConfigurationType]:
    return robot_kinematics.inverse_kinematics(X_B_TCP_target,
                                                 override_position_tolerance=[[0.002, 0.03, 0.01], [0.002, 0.03, 0.01]], 
                                                 override_rotation_tolerance=math.radians(20))
     
def show_trajectory_in_meshcat(trajectory):
    animate_joint_trajectory(
        meshcat,
        robot_diagram,
        arm_index,
        trajectory)

def _configuration_nearby(joint_trajectory: PathParameterizedTrajectory, manipulator: PositionManipulator) -> bool:
    """Assert that the first configuration of the trajectory is close to the current configuration of the manipulator."""
    current_configuration = manipulator.get_joint_configuration()
    first_configuration = joint_trajectory.value(0).squeeze()
    return np.isclose(first_configuration, current_configuration, atol=np.radians(1.0), rtol=0.0).all()

def execute_trajectory(joint_trajectory: PathParameterizedTrajectory, manipulator: PositionManipulator, duration_multiplier=1.0):
    assert joint_trajectory.start_time() == 0.0
    assert _configuration_nearby(joint_trajectory, manipulator)

    joint_trajectory.value(0).squeeze()

    period = 0.005  # Time per servo, approximately. This may be slightly changed because of rounding errors.
    # The period determines the times at which we sample the trajectory that was time-parameterized by TOPPRA.
    duration = (
        joint_trajectory.end_time() - joint_trajectory.start_time()
    )  # Should equal end_time(), but sanity check, or in case assertion is ever removed.

    n_servos = int(np.ceil(duration / period))
    period_adjusted = (duration * duration_multiplier) / n_servos  # can be slightly different from period due to rounding

    for t in np.linspace(0, duration, n_servos):
        joints = joint_trajectory.value(t).squeeze()
        logger.trace(
            f"Target joints: {np.rad2deg(joints)}, current joints: {np.rad2deg(manipulator.get_joint_configuration())} "
            f"(diff: {np.rad2deg(np.linalg.norm(joints - manipulator.get_joint_configuration())):0.02f} )"
        )
        manipulator.servo_to_joint_configuration(joints, period_adjusted).wait()

    # This avoids the abrupt stop and "thunk" sounds at the end of paths that end with non-zero velocity
    # However, I believe these functions are blocking, so right only stops after left has stopped.
    manipulator.rtde_control.servoStop(2.0)

    manipulator.move_to_joint_configuration(joints).wait()



In [ ]:
meshcat = Meshcat()

Let's move your robot to your home configuration. Make sure it does not collide.

In [ ]:
robot.move_to_joint_configuration(joints_home).wait()

<div class="alert alert-block alert-info"> <b>Task:</b>
Fill in all offsets and rotations depending on your setup. You have already done this in practical 3 but your setup might have changed.
    
</div>

In [ ]:
#TODO(@Student): fill in all offsets and rotations depending on your setup. You have done this for practical 3 already but your setup might have changed.

# Table -> Mounting plate of the robot
x_offset = 0.10
y_offset = 0.20

rpy_table_mountingplate = [0, 0, 0]
xyz_table_mountingplate = [x_offset, y_offset, 0.0]

X_Table_MountingPlate = RigidTransform(RollPitchYaw(*rpy_table_mountingplate), xyz_table_mountingplate)

# Table -> Rack 
x_offset = 0.2
y_offset = 0.5
z_offset = 0.5

roll = 0
pitch = 0
yaw = 0

X_Table_MountingRack = RigidTransform(RollPitchYaw(roll, pitch, yaw), [x_offset, y_offset, z_offset])

# Table -> Micro plate
x_offset = 0.1
y_offset = 0.1
z_offset = 0.1

roll = -np.pi/2
pitch = -np.pi/2
yaw = 0
X_Table_UPlate = RigidTransform(RollPitchYaw([roll, pitch, yaw]), [x_offset, y_offset, z_offset])

# Robot starting position
q_init = [0, -np.pi / 2, 0, -np.pi / 2, 0, 0] if robot is None else robot.get_joint_configuration()

In [ ]:
# Init env
robot_diagram_builder = RobotDiagramBuilder()
plant = robot_diagram_builder.plant()
parser = robot_diagram_builder.parser()
parser.SetAutoRenaming(True)

meshcat.Delete()
meshcat.Delete("/scenario")
meshcat.DeleteAddedControls()
add_meshcat(robot_diagram_builder, meshcat)

USE_PIPETTE = False
MOUNT_CAMERA = True

# 1. Find your models
table_urdf_path = str(Path(os.path.join("resources", "table.urdf")).resolve())
mounting_plate_urdf_path = str(Path(os.path.join("resources", "mounting_plate.urdf")).resolve())
arm_urdf_path = airo_models.get_urdf_path("ur3e")
gripper_urdf_path = airo_models.get_urdf_path("robotiq_2f_85")
gripper_urdf = airo_models.urdf.read_urdf(gripper_urdf_path)
make_static(gripper_urdf)
if USE_PIPETTE:
    gripper_urdf_path = str(Path(os.path.join("resources", "pipette.urdf")).resolve())
else:
    gripper_urdf_path = airo_models.urdf.write_urdf_to_tempfile(
        gripper_urdf, gripper_urdf_path, prefix="robotiq_2f_85_static_"
    )
if MOUNT_CAMERA:
    camera_urdf_path = airo_models.get_urdf_path("d435")
rack_urdf_path = str(Path(os.path.join("resources", "rack.urdf")).resolve())
uplate_urdf_path = str(Path(os.path.join("resources", "uplate.urdf")).resolve())

# 2. Add models to simulator
table_index = parser.AddModels(table_urdf_path)[0]
mounting_plate_index = parser.AddModels(mounting_plate_urdf_path)[0]
arm_index = parser.AddModels(arm_urdf_path)[0]
gripper_index = parser.AddModels(gripper_urdf_path)[0]
rack_index = parser.AddModels(rack_urdf_path)[0]
uplate_index = parser.AddModels(uplate_urdf_path)[0]
if MOUNT_CAMERA:
    wrist_camera_index = parser.AddModels(camera_urdf_path)[0]

# 3. Get mounting and attach frames for all models
world_frame = plant.world_frame()
table_frame = plant.GetFrameByName("base_link", table_index)
mounting_plate_frame = plant.GetFrameByName("base_link", mounting_plate_index)
mounting_plate_top_centre_frame = plant.GetFrameByName("plate_top_centre",
                                                       mounting_plate_index)  # this is a virtual helper frame 
arm_frame = plant.GetFrameByName("base_link", arm_index)
arm_tool_frame = plant.GetFrameByName("tool0", arm_index)
gripper_frame = plant.GetFrameByName("base_link", gripper_index)
rack_frame = plant.GetFrameByName("base_link", rack_index)
uplate_frame = plant.GetFrameByName("base_link", uplate_index)
if MOUNT_CAMERA:
    wrist_camera_frame =  plant.GetFrameByName("base_link", wrist_camera_index)

# 4. Define relative transform, i.e. specify child-parent transforms
X_MountingPlateTopCentre_ArmBaseLink = RigidTransform(RollPitchYaw(0, 0, -np.pi / 2), [0, 0, 0])
X_Tool0_GripperBase = RigidTransform(RollPitchYaw(0, 0, 0), [0, 0, 0])
if MOUNT_CAMERA:
    with open("camera_pose.json", "r") as f:
        X_Tcp_WristCam = RigidTransform(Pose.model_validate_json(f.read()).as_homogeneous_matrix())

# Add TCP frame
tcp_offset = 0.05 if USE_PIPETTE else 0.175
# TCP is between the fingers and tcp_offset cm from the wrist towards the gripper forwards direction (Z+ in TCP frame)
X_Tool0Tcp = RigidTransform(RotationMatrix.Identity(), [0, 0, tcp_offset])
arm_tcp_frame = plant.AddFrame(FixedOffsetFrame("tcp", plant.GetFrameByName("tool0", arm_index), X_Tool0Tcp))

# 5. Weld frames together
plant.WeldFrames(world_frame, table_frame)  # world -> table
plant.WeldFrames(table_frame, mounting_plate_frame, X_Table_MountingPlate)  # table -> mountingplate
plant.WeldFrames(mounting_plate_top_centre_frame, arm_frame,
                 X_MountingPlateTopCentre_ArmBaseLink)  # mountingplate -> arm
plant.WeldFrames(arm_tool_frame, gripper_frame, X_Tool0_GripperBase)  # arm -> gripper 
plant.WeldFrames(table_frame, rack_frame, X_Table_MountingRack)  # table -> rack
plant.WeldFrames(table_frame, uplate_frame, X_Table_UPlate)  # table -> uplate
if MOUNT_CAMERA:
    plant.WeldFrames(arm_tcp_frame, wrist_camera_frame, X_Tcp_WristCam)  # table -> uplate

# Finish the build of the simulation env
robot_diagram, context = finish_build(robot_diagram_builder, meshcat)

# robot arm init joint positions
plant_context = plant.GetMyContextFromRoot(context)
plant.SetPositions(plant_context, arm_index, q_init)
robot_diagram.ForcedPublish(context)

robot_kinematics = RobotKinematics(robot_diagram, arm_index, meshcat)

<div class="alert alert-block alert-warning"> 
Check Meshcat to check whether your simulated environment looks similar to your real environment!    
</div>

In [ ]:
# Setup collision checker for planner 
collision_checker = SceneGraphCollisionChecker(
    model=robot_diagram,
    robot_model_instances=[arm_index, gripper_index],
    edge_step_size=0.125,  # Arbitrary value: we don't use the CheckEdgeCollisionFree
    env_collision_padding=0.005,
    self_collision_padding=0.005,
)

In [ ]:
# setup planner
planner = SingleArmOmplPlanner(collision_checker.CheckConfigCollisionFree, inverse_kinematics_fn=inverse_kinematics_fn)

In [ ]:
# lets demonstrate the origin of the world, which is the origin of the table (base_link)
table = plant.GetModelInstanceByName("table")
X_W_T = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(), plant.GetBodyByName("base_link", table))
visualize_frame(meshcat, "/scenario/planner_origin", X_W_T, length=0.35, radius=0.01)
X_W_B = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                  plant.GetBodyByName("base", plant.GetModelInstanceByName("ur3e")))
visualize_frame(meshcat, "/scenario/arm_base", X_W_B, length=0.20, radius=0.005)

In [ ]:
# plan this grasp using X_base_pregrasp  
robot_base = plant.GetBodyByName("base", plant.GetModelInstanceByName("ur3e"))
X_W_B = plant.EvalBodyPoseInWorld(plant_context, robot_base)
X_W_TcpPreGrasp = X_W_B @ RigidTransform(X_base_pregrasp)
visualize_frame(meshcat, "/scenario/alignment_block/X_W_TcpPreGrasp", X_W_TcpPreGrasp, length=0.15, radius=0.005,)

<div class="alert alert-block alert-info"> <b>Task:</b> Run the next cell to run the motion planner in simulation. </div>

In [ ]:
q_init = robot.get_joint_configuration()
path = planner.plan_to_tcp_pose(q_init, X_base_pregrasp)
trajectory = time_parametrize_toppra(plant, path)
show_trajectory_in_meshcat(trajectory)
print(f"\nIf planning was successful, you can now go to {meshcat.web_url()} and look at the planning result. You can open the drop-down in the righttop of the screen, search Animations, and drag the time slider to review the trajectory.")

<div class="alert alert-block alert-info"> <b>Task:</b>  Now execute the trajectory on the real robot. We use a combination of execute_trajectory(q_traj) followed by a robot.move_to_tcp_pose(X_B_Tcp). Why? </div>

Your answer: 

In [ ]:
gripper.open().wait()
if robot is not None:
    execute_trajectory(trajectory, robot, duration_multiplier=2)

In [ ]:
if len(_inverse_kinematics_analytical(X_base_TcpGrasp)) == 0:
    print("There is probably no good real solution to reach the eventual grasping pose you specified. Consider changing X_base_TcpGrasp, the position of your microplate or the setup of your robot arm or rack.")

In [ ]:
robot.move_to_tcp_pose(X_base_pregrasp)

In [ ]:
gripper.move(0.03, speed=0.002, force=0.001).wait()
gripper.move(0.035, speed=0.002, force=0.001).wait()

In [ ]:
robot.move_to_tcp_pose(X_base_TcpGrasp).wait()  
gripper.close().wait()

In [ ]:
robot.move_to_tcp_pose(X_base_retreat).wait()

# The end
This practical was the pinnacle of the IRM course: you combined all the knowledge from previous courses and practicals to construct your first intelligent robot pipeline. You have used both analytical and optimization-based inverse kinematics to find a q_target for a target pose to grasp the microplate which was found by a deep learning based keypoint detector for which we translated the keypoints in image space to workspace using calibrated cameras. 
There is one more practical remaining in which we will drive this integration even further. See you then!   

Before shutting down the robot, please use freedrive to put it in a safe position (away from objects and do not let any links of robots rest outside the table so people do not collide against the robot when passing).